#Extracting Attention Maps from ViT


In [12]:
import torch
from torchvision.models import vit_b_16, ViT_B_16_Weights

In [17]:
def extract_attention_maps(image_tensor):
  #Load pretrained ViT model
  model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
  model.eval()
  #Register a forward hook to capture attention maps
  attention_maps = []

  def hook_fn(module, input, output):
    attention_maps.append(output[1])
  last_attn_block = model.encoder.layers[-1].self_attention
  hook_handle = last_attn_block.register_forward_hook(hook_fn)

  #Add batch dimension
  batched_image = image_tensor.unsqueeze(0)

  #Pass the image through the model
  with torch.no_grad():
    _ = model(batched_image)

  #Remove the hook
  hook_handle.remove()
  attention_weights = attention_maps[0][0]
  return attention_weights

image_tensor = torch.randn(3, 224, 224)
attention_weights = extract_attention_maps(image_tensor)
print(attention_weights.shape)

TypeError: 'NoneType' object is not subscriptable

In [18]:
import torch
from torchvision.models import vit_b_16, ViT_B_16_Weights

def extract_attention_maps(image_tensor):
    # Load pretrained ViT model
    model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
    model.eval()

    attention_maps = []

    def hook_fn(module, input, output):
        # input: tuple with a single tensor of shape [seq_len, batch, embed_dim]
        qkv = module.in_proj_weight  # [3 * embed_dim, embed_dim]
        q_proj = module.in_proj_weight[:module.embed_dim, :]
        k_proj = module.in_proj_weight[module.embed_dim:2 * module.embed_dim, :]
        v_proj = module.in_proj_weight[2 * module.embed_dim:, :]

        q_bias = module.in_proj_bias[:module.embed_dim]
        k_bias = module.in_proj_bias[module.embed_dim:2 * module.embed_dim]

        # Get input tensor to attention: [seq_len, batch, embed_dim]
        x = input[0]  # [seq_len, batch, embed_dim]
        x = x.permute(1, 0, 2)  # [batch, seq_len, embed_dim]

        # Linear projection
        q = torch.nn.functional.linear(x, q_proj, q_bias)
        k = torch.nn.functional.linear(x, k_proj, k_bias)

        # Split into heads
        B, N, C = q.shape
        num_heads = module.num_heads
        head_dim = C // num_heads

        q = q.view(B, N, num_heads, head_dim).transpose(1, 2)  # [B, heads, N, head_dim]
        k = k.view(B, N, num_heads, head_dim).transpose(1, 2)  # [B, heads, N, head_dim]

        # Compute attention
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
        attn_weights = torch.softmax(attn_scores, dim=-1)  # [B, heads, N, N]

        attention_maps.append(attn_weights)

    # Register hook on the last attention layer
    last_attn_block = model.encoder.layers[-1].self_attention
    hook_handle = last_attn_block.register_forward_hook(hook_fn)

    # Add batch dimension
    batched_image = image_tensor.unsqueeze(0)

    # Forward pass
    with torch.no_grad():
        _ = model(batched_image)

    # Remove hook
    hook_handle.remove()

    return attention_maps[0]  # [1, heads, tokens, tokens]

# Example usage
image_tensor = torch.randn(3, 224, 224)
attention_weights = extract_attention_maps(image_tensor)
print(attention_weights.shape)  # Expected: [1, 12, 197, 197]


torch.Size([197, 12, 1, 1])
